### Install Vertex AI SDK for Python and other required packages



In [1]:
# Vertex SDK for Python
! pip3 install --upgrade --quiet  google-cloud-aiplatform


[notice] A new release of pip is available: 25.2 -> 25.3
[notice] To update, run: pip install --upgrade pip


### Set Google Cloud project information
Learn more about [setting up a project and a development environment](https://cloud.google.com/vertex-ai/docs/start/cloud-environment).

In [2]:
PROJECT_ID = "mlops-ga-474200"  # @param {type:"string"}
LOCATION = "us-central1"  # @param {type:"string"}

### Create a Cloud Storage bucket

Create a storage bucket to store intermediate artifacts such as datasets.

In [3]:
DATA_VERSION = "v1"
BUCKET_URI = f"gs://ga5-bucket"  # @param {type:"string"}
MODEL_ARTIFACT_DIR=f"iris_classifier/{DATA_VERSION}_model"

**If your bucket doesn't already exist**: Run the following cell to create your Cloud Storage bucket.

In [4]:
! gsutil mb -l {LOCATION} -p {PROJECT_ID} {BUCKET_URI}

Creating gs://ga5-bucket/...
ServiceException: 409 A Cloud Storage bucket named 'ga5-bucket' already exists. Try another name. Bucket names must be globally unique across all Google Cloud projects, including those outside of your organization.


### Initialize Vertex AI SDK for Python

To get started using Vertex AI, you must have an existing Google Cloud project and [enable the Vertex AI API](https://console.cloud.google.com/flows/enableapi?apiid=aiplatform.googleapis.com).

In [5]:
from google.cloud import aiplatform

aiplatform.init(project=PROJECT_ID, location=LOCATION, staging_bucket=BUCKET_URI)

/opt/conda/lib/python3.10/site-packages/google/cloud/aiplatform/models.py:52: FutureWarning: Support for google-cloud-storage < 3.0.0 will be removed in a future version of google-cloud-aiplatform. Please upgrade to google-cloud-storage >= 3.0.0.
  from google.cloud.aiplatform.utils import gcs_utils


### Import the required libraries

In [6]:
import os
import sys

### Set up MLFlow

In [7]:
import mlflow
from mlflow import MlflowClient
from mlflow.models import infer_signature
from pprint import pprint

mlflow.set_tracking_uri("http://127.0.0.1:8100")
client = MlflowClient(mlflow.get_tracking_uri())
all_experiments = client.search_experiments()
print(all_experiments)

[<Experiment: artifact_location='mlflow-artifacts:/659293658728175616', creation_time=1761924322499, experiment_id='659293658728175616', last_update_time=1761924322499, lifecycle_stage='active', name='IRIS classifier: GA5 Hands On', tags={'mlflow.experimentKind': 'custom_model_development'}>, <Experiment: artifact_location='mlflow-artifacts:/0', creation_time=1761908128609, experiment_id='0', last_update_time=1761908128609, lifecycle_stage='active', name='Default', tags={}>]


In [8]:
mlflow.set_experiment("iris-poisoning-experiments")

2025/11/16 16:03:34 INFO mlflow.tracking.fluent: Experiment with name 'iris-poisoning-experiments' does not exist. Creating a new experiment.


<Experiment: artifact_location='mlflow-artifacts:/501387673414268516', creation_time=1763309014622, experiment_id='501387673414268516', last_update_time=1763309014622, lifecycle_stage='active', name='iris-poisoning-experiments', tags={}>

In [9]:
# src/poisoning.py
import numpy as np
import pandas as pd

def poison_features(df: pd.DataFrame, rate: float, random_state: int = 42, feature_cols=None):
    """
    Replace features in `feature_cols` for a random subset of rows at fraction `rate`
    with random values drawn from a sensible range (uniform over feature ranges).
    - df: DataFrame including features and label column 'species' (or target name)
    - rate: float in (0,1) fraction of rows to poison
    - feature_cols: list of columns to poison; if None, numeric columns except target are used
    Returns a new DataFrame (copy).
    """
    if not 0 <= rate <= 1:
        raise ValueError("rate must be between 0 and 1")
    df = df.copy()
    rng = np.random.RandomState(random_state)
    if feature_cols is None:
        # default: numeric columns except last (assumed label)
        numeric_cols = df.select_dtypes(include=[np.number]).columns.tolist()
        feature_cols = numeric_cols

    n = len(df)
    k = int(np.round(rate * n))
    if k == 0:
        return df

    # choose rows to poison
    idx = rng.choice(df.index, size=k, replace=False)

    # sample random values per column using uniform between min-1*std and max+1*std
    for col in feature_cols:
        col_min, col_max = df[col].min(), df[col].max()
        col_std = df[col].std() if df[col].std() > 0 else 1.0
        low = col_min - col_std
        high = col_max + col_std
        df.loc[idx, col] = rng.uniform(low=low, high=high, size=k)

    return df

def flip_labels(df: pd.DataFrame, rate: float, label_col='species', random_state=42):
    """
    Flip labels for a fraction rate of the rows (uniformly choose a wrong label).
    Useful if you also want to run label-flip attacks.
    """
    df = df.copy()
    rng = np.random.RandomState(random_state)
    n = len(df)
    k = int(np.round(rate * n))
    if k == 0:
        return df

    idx = rng.choice(df.index, size=k, replace=False)
    classes = df[label_col].unique().tolist()
    for i in idx:
        original = df.at[i, label_col]
        choices = [c for c in classes if c != original]
        df.at[i, label_col] = rng.choice(choices)
    return df


In [10]:
# src/train_poison_mlflow.py
import os
import mlflow
import joblib
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, confusion_matrix, classification_report
from sklearn.model_selection import train_test_split
import pandas as pd
import numpy as np

In [11]:
TARGET_COL = "species"

In [17]:
def train_and_eval(df, poison_rate=0.0, poison_mode="features", seed=42, run_name=None):
    """
    Trains and evaluates a simple pipeline and returns metrics + artifacts dict.
    Also logs to MLflow when mlflow.active_run() is present.
    """
    # poison copy of data
    if poison_mode == "features":
        dfp = poison_features(df, poison_rate, random_state=seed)
    elif poison_mode == "label-flip":
        dfp = flip_labels(df, poison_rate, label_col=TARGET_COL, random_state=seed)
    else:
        raise ValueError("poison_mode must be 'features' or 'label-flip'")

    # features and labels
    X = dfp.drop(columns=[TARGET_COL])
    y = dfp[TARGET_COL]

    X_train, X_test, y_train, y_test = train_test_split(
        X, y, stratify=y, test_size=0.2, random_state=seed
    )

    pipe = Pipeline([
        ("scaler", StandardScaler()),
        ("clf", LogisticRegression(max_iter=500, random_state=seed))
    ])

    pipe.fit(X_train, y_train)
    y_pred = pipe.predict(X_test)

    acc = accuracy_score(y_test, y_pred)
    prec, rec, f1, _ = precision_recall_fscore_support(y_test, y_pred, average='macro', zero_division=0)

    cm = confusion_matrix(y_test, y_pred, labels=np.unique(y))
    cr = classification_report(y_test, y_pred, zero_division=0)

    # prepare artifacts: confusion matrix plot
    plt.figure(figsize=(5,4))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=np.unique(y), yticklabels=np.unique(y))
    plt.xlabel("predicted")
    plt.ylabel("true")
    plt.title(f"Confusion matrix (poison={poison_rate})")
    cm_path = f"artifacts/cm_poison_{int(poison_rate*100)}.png"
    plt.tight_layout()
    plt.savefig(cm_path)
    plt.close()

    # Save model locally
    model_path = f"artifacts/model_poison_{int(poison_rate*100)}.pkl"
    joblib.dump(pipe, model_path)

    metrics = {"accuracy": acc, "precision_macro": prec, "recall_macro": rec, "f1_macro": f1}
    artifacts = {"confusion_matrix": cm_path, "model": model_path, "classification_report": cr}

    return metrics, artifacts, pipe, (X_test, y_test, y_pred)

def run_experiments(csv_path, poison_levels=(0.0, 0.05, 0.10, 0.50), poison_mode="features", seed=42):
    df = pd.read_csv(csv_path)
    results = []
    for p in poison_levels:
        with mlflow.start_run(run_name=f"poison_{int(p*100)}") as run:
            mlflow.log_param("poison_rate", p)
            mlflow.log_param("poison_mode", poison_mode)
            mlflow.log_param("seed", seed)

            metrics, artifacts, model, (X_test, y_test, y_pred) = train_and_eval(df, poison_rate=p, poison_mode=poison_mode, seed=seed)
            # log metrics
            for k, v in metrics.items():
                mlflow.log_metric(k, float(v))

            # log classification report as artifact
            cr_text = artifacts["classification_report"]
            cr_path = f"artifacts/classification_report_{int(p*100)}.txt"
            with open(cr_path, "w") as fh:
                fh.write(cr_text)
            mlflow.log_artifact(cr_path)

            # log confusion matrix image and model
            mlflow.log_artifact(artifacts["confusion_matrix"])
            mlflow.log_artifact(artifacts["model"])


            print(f"[run={run.info.run_id}] poison={p} metrics={metrics}")
            results.append((p, metrics, run.info.run_id))
    return results


In [18]:
csv_path = BUCKET_URI+f'/data/{DATA_VERSION}/data.csv'
poison_levels="0.0,0.05,0.1,0.5"
levels = [float(x) for x in poison_levels.split(",")]
mode="features" #label-flip
seed=42

In [19]:
res = run_experiments(csv_path, poison_levels=levels, poison_mode=mode, seed=seed)
print("Finished experiments:", res)

[run=aa79997b7c074ebe87229918ac919a74] poison=0.0 metrics={'accuracy': 0.9523809523809523, 'precision_macro': 0.9583333333333334, 'recall_macro': 0.9444444444444445, 'f1_macro': 0.9474747474747476}
🏃 View run poison_0 at: http://127.0.0.1:8100/#/experiments/501387673414268516/runs/aa79997b7c074ebe87229918ac919a74
🧪 View experiment at: http://127.0.0.1:8100/#/experiments/501387673414268516
[run=501eceb172ce400483dd193d94e32d5d] poison=0.05 metrics={'accuracy': 0.9523809523809523, 'precision_macro': 0.9583333333333334, 'recall_macro': 0.9444444444444445, 'f1_macro': 0.9474747474747476}
🏃 View run poison_5 at: http://127.0.0.1:8100/#/experiments/501387673414268516/runs/501eceb172ce400483dd193d94e32d5d
🧪 View experiment at: http://127.0.0.1:8100/#/experiments/501387673414268516
[run=c1825bd387a849d28bf1afae6f4b6051] poison=0.1 metrics={'accuracy': 0.8095238095238095, 'precision_macro': 0.8583333333333334, 'recall_macro': 0.7916666666666666, 'f1_macro': 0.7883986928104575}
🏃 View run poison

In [20]:
!gsutil cp -r artifacts/ {BUCKET_URI}/ga-8/{MODEL_ARTIFACT_DIR}/

Copying file://artifacts/classification_report_0.txt [Content-Type=text/plain]...
Copying file://artifacts/model_poison_0.pkl [Content-Type=application/octet-stream]...
Copying file://artifacts/model_poison_50.pkl [Content-Type=application/octet-stream]...
Copying file://artifacts/classification_report_50.txt [Content-Type=text/plain]...
- [4 files][  4.7 KiB/  4.7 KiB]                                                
==> NOTE: You are performing a sequence of gsutil operations that may
run significantly faster if you instead use gsutil -m cp ... Please
see the -m section under "gsutil help options" for further information
about when gsutil -m can be advantageous.

Copying file://artifacts/model_poison_10.pkl [Content-Type=application/octet-stream]...
Copying file://artifacts/cm_poison_10.png [Content-Type=image/png]...           
Copying file://artifacts/cm_poison_5.png [Content-Type=image/png]...            
Copying file://artifacts/classification_report_5.txt [Content-Type=text/plain